# Phase 1 — Data Acquisition & Inspection

**Dataset:** CICIDS2017 (Canadian Institute for Cybersecurity Intrusion Detection Evaluation Dataset 2017)

## Setup
Before running this notebook, download the data:
```bash
python scripts/download_data.py
```

---

## Day → Attack mapping

| File | Day | Attack types |
|------|-----|--------------|
| `Monday-WorkingHours.pcap_ISCX.csv` | Monday | BENIGN only (baseline traffic) |
| `Tuesday-WorkingHours.pcap_ISCX.csv` | Tuesday | FTP-Patator, SSH-Patator |
| `Wednesday-workingHours.pcap_ISCX.csv` | Wednesday | DoS Hulk, DoS GoldenEye, Heartbleed, DoS slowloris, DoS Slowhttptest |
| `Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv` | Thursday AM | Web Attack – Brute Force, XSS, SQL Injection |
| `Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv` | Thursday PM | Infiltration |
| `Friday-WorkingHours-Morning.pcap_ISCX.csv` | Friday AM | Bot |
| `Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv` | Friday PM | PortScan |
| `Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv` | Friday PM | DDoS |

> **Label column:** ` Label` (note the leading space — a known CICIDS2017 quirk, handled in Phase 2)

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/raw")

FILES = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
]

missing = [f for f in FILES if not (DATA_DIR / f).exists()]
if missing:
    print("Missing files — run: python scripts/download_data.py")
    for f in missing:
        print(f"  {f}")
else:
    print("All 8 files found.")

## Summary table — all 8 files

In [ ]:
summary = []
for fname in FILES:
    path = DATA_DIR / fname
    df = pd.read_csv(path, low_memory=False)
    # Detect label column (may have leading space)
    label_col = next((c for c in df.columns if c.strip() == "Label"), None)
    summary.append({
        "file": fname,
        "rows": len(df),
        "columns": len(df.columns),
        "label_col": repr(label_col),
        "unique_labels": df[label_col].nunique() if label_col else "N/A",
    })

pd.DataFrame(summary)

## Column inspection (first file)

In [ ]:
df = pd.read_csv(DATA_DIR / FILES[0], low_memory=False)

print("Raw column names (first 10):")
print(df.columns[:10].tolist())

cols_with_space = [c for c in df.columns if c != c.strip()]
print(f"\nColumns with leading/trailing spaces: {len(cols_with_space)}")
print(cols_with_space[:5])

print("\nDtypes:")
print(df.dtypes.value_counts())

df.head(3)

## Label distribution per file

Shows BENIGN vs attack counts → reveals the class imbalance that Phase 2 will address.

In [ ]:
for fname in FILES:
    df = pd.read_csv(DATA_DIR / fname, low_memory=False)
    label_col = next((c for c in df.columns if c.strip() == "Label"), None)
    if label_col:
        print(f"\n{'='*60}")
        print(f"{fname}")
        print(df[label_col].value_counts().to_string())

## Known pitfalls — preview for Phase 2

| Issue | Description | Fix (Phase 2) |
|-------|-------------|---------------|
| Leading spaces in column names | e.g. `' Label'` instead of `'Label'` | `df.columns = df.columns.str.strip()` |
| `Infinity` values | Some flow features (like `Flow Bytes/s`) overflow | `df.replace([np.inf, -np.inf], np.nan)` then drop/clip |
| `NaN` values | Missing entries in flow duration features | `df.dropna()` or `df.fillna()` |
| Class imbalance | BENIGN >> attacks (often 80-90% BENIGN) | `class_weight='balanced'` / undersampling / SMOTE |

In [ ]:
import numpy as np

# Check Infinity and NaN in first file
df = pd.read_csv(DATA_DIR / FILES[0], low_memory=False)
numeric_df = df.select_dtypes(include=[np.number])

inf_cols = numeric_df.columns[numeric_df.isin([np.inf, -np.inf]).any()].tolist()
nan_cols = numeric_df.columns[numeric_df.isna().any()].tolist()

print(f"Columns with Infinity: {inf_cols}")
print(f"Columns with NaN:      {nan_cols}")